# BBQ 데이터셋 확인

Kaggle Dataset에 올라간 BBQ 변환 데이터를 확인합니다.

**실행 전**: Add data → `skku-bbq-data` 추가

In [ ]:
import json
from pathlib import Path

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── 실제 경로 자동 탐색 ─────────────────────────────────────────
print("=== /kaggle/input 전체 구조 ===")
for p in sorted(Path("/kaggle/input").rglob("train.csv")):
    print(f"  {p}")

# train.csv가 있는 디렉토리를 BBQ_DATA_DIR로 설정
candidates = list(Path("/kaggle/input").rglob("train.csv"))
if not candidates:
    raise FileNotFoundError("/kaggle/input 아래에 train.csv가 없습니다. Dataset이 추가됐는지 확인하세요.")

BBQ_DATA_DIR = candidates[0].parent
print(f"\n→ 사용할 경로: {BBQ_DATA_DIR}")

train_df = pd.read_csv(BBQ_DATA_DIR / "train.csv")
val_df   = pd.read_csv(BBQ_DATA_DIR / "val.csv")

print(f"\nTrain: {len(train_df):,}")
print(f"Val  : {len(val_df):,}")

## 기본 정보

In [ ]:
print("[컬럼]")
print(train_df.columns.tolist())
print()
print("[결측치]")
print(train_df.isnull().sum())

## 분포 확인

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train_df["label"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Label 분포", color="steelblue"
)
train_df["context_condition"].value_counts().plot(
    kind="bar", ax=axes[1], title="Context Condition", color="orange"
)
train_df["category"].value_counts().plot(
    kind="barh", ax=axes[2], title="카테고리 분포", color="green"
)

for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("[Label 분포]")
print(train_df["label"].value_counts().to_string())
print()
print("[Context Condition]")
print(train_df["context_condition"].value_counts().to_string())

## 샘플 확인

In [ ]:
def show_sample(row):
    answers = json.loads(row["answers"])
    print(f"ID       : {row['sample_id']}")
    print(f"Category : {row['category']}")
    print(f"Condition: {row['context_condition']}")
    print(f"Context  : {row['context']}")
    print(f"Question : {row['question']}")
    print(f"Options  : 0. {answers[0]}")
    print(f"           1. {answers[1]}")
    print(f"           2. {answers[2]}")
    print(f"Label    : {row['label']} → {answers[int(row['label'])]}")
    print("-" * 60)

# disambig 샘플 (근거 있음)
print("=== Disambiguated (근거 있음) ===")
for _, row in train_df[train_df["context_condition"] == "disambig"].head(3).iterrows():
    show_sample(row)

print()
# ambig 샘플 (근거 없음)
print("=== Ambiguous (근거 없음) ===")
for _, row in train_df[train_df["context_condition"] == "ambig"].head(3).iterrows():
    show_sample(row)

## 이미지 확인 (Placeholder)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for i, ax in enumerate(axes):
    img_rel = train_df.iloc[i]["image_path"].lstrip("./")
    img = mpimg.imread(BBQ_DATA_DIR / img_rel)
    ax.imshow(img)
    ax.set_title(f"Sample {i}")
    ax.axis("off")
plt.suptitle("Placeholder 이미지 (224x224 회색)")
plt.tight_layout()
plt.show()

sample_img = Image.open(BBQ_DATA_DIR / train_df.iloc[0]["image_path"].lstrip("./"))
print(f"이미지 크기: {sample_img.size}, 모드: {sample_img.mode}")

## 파인튜닝 노트북과 경로 일치 확인

In [ ]:
print("파인튜닝 노트북에 입력할 경로:")
print(f'  BBQ_DATA_DIR = "{BBQ_DATA_DIR}"')
print(f'  TRAIN_CSV    = "{BBQ_DATA_DIR}/train.csv"')
print(f'  VAL_CSV      = "{BBQ_DATA_DIR}/val.csv"')

# 파일 존재 여부 최종 확인
for p in [BBQ_DATA_DIR / "train.csv", BBQ_DATA_DIR / "val.csv", BBQ_DATA_DIR / "images"]:
    status = "✓" if p.exists() else "✗"
    print(f"  [{status}] {p}")